# CPU Scheduling Simulator

Simulates six CPU scheduling techniques and displays a **Gantt chart** (process on Y-axis, time on X-axis).

| Key | Algorithm | Mode |
|-----|-----------|------|
| `FCFS` | First Come First Served | Non-preemptive |
| `SJF` | Shortest Job First | Non-preemptive |
| `SRTF` | Shortest Remaining Time First | Preemptive |
| `RR` | Round Robin | Non-preemptive (time-sliced) |
| `PRIORITY` | Priority Scheduling | Non-preemptive |
| `PRIORITY_P` | Priority Scheduling | Preemptive |
| `MLQ` | Multilevel Queue | Q0=FCFS, Q1=RR, Q2=FCFS |

**CSV columns:** `pid, arrival_time, burst_time, priority, queue_level`
- `priority`: lower number = higher priority
- `queue_level`: 0 (highest) … 2 (lowest), used by MLQ only


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np
from copy import deepcopy
from IPython.display import display, HTML

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10


In [7]:
df = pd.read_csv('processes.csv')
print("Processes loaded from CSV:")
display(df)


Processes loaded from CSV:


,pid,arrival_time,burst_time,priority,queue_level
0,P1,0,8,3,1
1,P2,1,4,1,0
2,P3,2,9,4,2
3,P4,3,5,2,1
4,P5,4,2,5,0
5,P6,5,6,1,2
6,P7,6,3,3,1
7,P8,7,1,2,0


In [8]:
# ── Helper ────────────────────────────────────────────────────────────────────
def _merge(tl):
    """Merge adjacent same-process segments into one bar."""
    if not tl:
        return []
    m = [list(tl[0])]
    for pid, s, e in tl[1:]:
        if m[-1][0] == pid and m[-1][2] == s:
            m[-1][2] = e
        else:
            m.append([pid, s, e])
    return [tuple(x) for x in m]


def fcfs(procs):
    """First Come First Served (non-preemptive)."""
    tl, t = [], 0
    for p in sorted(procs, key=lambda x: (x['arrival_time'], x['pid'])):
        t = max(t, p['arrival_time'])
        tl.append((p['pid'], t, t + p['burst_time']))
        t += p['burst_time']
    return tl


def sjf(procs):
    """Shortest Job First (non-preemptive)."""
    tl, t = [], 0
    rem = sorted(procs, key=lambda x: x['arrival_time'])
    while rem:
        ready = [p for p in rem if p['arrival_time'] <= t]
        if not ready:
            t = min(p['arrival_time'] for p in rem)
            ready = [p for p in rem if p['arrival_time'] <= t]
        p = min(ready, key=lambda x: (x['burst_time'], x['pid']))
        rem.remove(p)
        tl.append((p['pid'], t, t + p['burst_time']))
        t += p['burst_time']
    return tl


def srtf(procs):
    """Shortest Remaining Time First (preemptive SJF). Event-driven."""
    rb  = {p['pid']: p['burst_time']   for p in procs}
    ar  = {p['pid']: p['arrival_time'] for p in procs}
    ids = [p['pid'] for p in procs]
    t   = min(ar.values())
    done, cur, cs, tl = set(), None, t, []

    while len(done) < len(ids):
        ready = [p for p in ids if p not in done and ar[p] <= t]
        if not ready:
            if cur and t > cs:
                tl.append((cur, cs, t)); cur = None
            t = min(ar[p] for p in ids if p not in done)
            continue
        nxt = min(ready, key=lambda p: (rb[p], p))
        if nxt != cur:
            if cur and t > cs:
                tl.append((cur, cs, t))
            cur, cs = nxt, t
        future   = [ar[p] for p in ids if p not in done and ar[p] > t]
        next_arr = min(future) if future else float('inf')
        if next_arr < t + rb[cur]:
            rb[cur] -= next_arr - t; t = next_arr
            if rb[cur] == 0:
                done.add(cur); tl.append((cur, cs, t)); cur = None
        else:
            t += rb[cur]; rb[cur] = 0
            done.add(cur); tl.append((cur, cs, t)); cur = None
    return _merge(tl)


def round_robin(procs, quantum=2):
    """Round Robin with configurable time quantum."""
    rb     = {p['pid']: p['burst_time']   for p in procs}
    ar     = {p['pid']: p['arrival_time'] for p in procs}
    by_arr = sorted(procs, key=lambda x: (x['arrival_time'], x['pid']))
    arrived, queue, tl = set(), [], []
    t = by_arr[0]['arrival_time']

    def _enq(up_to):
        for p in by_arr:
            if p['pid'] not in arrived and ar[p['pid']] <= up_to:
                arrived.add(p['pid']); queue.append(p['pid'])

    _enq(t)
    while queue or any(p['pid'] not in arrived for p in by_arr):
        if not queue:
            t = min(ar[p['pid']] for p in by_arr if p['pid'] not in arrived)
            _enq(t)
        pid = queue.pop(0)
        run = min(quantum, rb[pid])
        tl.append((pid, t, t + run))
        rb[pid] -= run; t += run
        _enq(t)
        if rb[pid] > 0:
            queue.append(pid)
    return _merge(tl)


def priority_np(procs):
    """Priority scheduling (non-preemptive). Lower number = higher priority."""
    tl, t = [], 0
    rem = list(procs)
    while rem:
        ready = [p for p in rem if p['arrival_time'] <= t]
        if not ready:
            t = min(p['arrival_time'] for p in rem)
            ready = [p for p in rem if p['arrival_time'] <= t]
        p = min(ready, key=lambda x: (x['priority'], x['arrival_time']))
        rem.remove(p)
        tl.append((p['pid'], t, t + p['burst_time']))
        t += p['burst_time']
    return tl


def priority_p(procs):
    """Priority scheduling (preemptive). Lower number = higher priority."""
    rb  = {p['pid']: p['burst_time']   for p in procs}
    ar  = {p['pid']: p['arrival_time'] for p in procs}
    pri = {p['pid']: p['priority']     for p in procs}
    ids = [p['pid'] for p in procs]
    t   = min(ar.values())
    done, cur, cs, tl = set(), None, t, []

    while len(done) < len(ids):
        ready = [p for p in ids if p not in done and ar[p] <= t]
        if not ready:
            if cur and t > cs:
                tl.append((cur, cs, t)); cur = None
            t = min(ar[p] for p in ids if p not in done)
            continue
        nxt = min(ready, key=lambda p: (pri[p], ar[p]))
        if nxt != cur:
            if cur and t > cs:
                tl.append((cur, cs, t))
            cur, cs = nxt, t
        future   = [ar[p] for p in ids if p not in done and ar[p] > t]
        next_arr = min(future) if future else float('inf')
        if next_arr < t + rb[cur]:
            rb[cur] -= next_arr - t; t = next_arr
            if rb[cur] == 0:
                done.add(cur); tl.append((cur, cs, t)); cur = None
        else:
            t += rb[cur]; rb[cur] = 0
            done.add(cur); tl.append((cur, cs, t)); cur = None
    return _merge(tl)


def multilevel_queue(procs, quantum=2):
    """
    Multilevel Queue (tick-by-tick simulation):
      Queue 0 — highest priority — FCFS
      Queue 1 — medium  priority — Round Robin (quantum)
      Queue 2 — lowest  priority — FCFS
    A non-empty higher-priority queue always preempts lower queues.
    """
    rb     = {p['pid']: p['burst_time']   for p in procs}
    ar     = {p['pid']: p['arrival_time'] for p in procs}
    ql     = {p['pid']: p['queue_level']  for p in procs}
    by_arr = sorted(procs, key=lambda x: (x['arrival_time'], x['pid']))
    queues = {0: [], 1: [], 2: []}
    arrived, rr_rem, done, ticks = set(), {}, set(), []
    t     = by_arr[0]['arrival_time']
    max_t = t + sum(p['burst_time'] for p in procs) + 5

    def _arrive(up_to):
        for p in by_arr:
            pid = p['pid']
            if pid not in arrived and ar[pid] <= up_to:
                arrived.add(pid)
                queues[ql[pid]].append(pid)
                if ql[pid] == 1:
                    rr_rem[pid] = quantum

    _arrive(t)
    while len(done) < len(procs) and t < max_t:
        _arrive(t)
        run_q = next((q for q in [0, 1, 2] if queues[q]), None)
        if run_q is None:
            t += 1; continue
        pid = queues[run_q][0]
        ticks.append((pid, t))
        rb[pid] -= 1
        if run_q == 1:
            rr_rem[pid] -= 1
        t += 1
        if rb[pid] == 0:
            done.add(pid); queues[run_q].pop(0)
            if run_q == 1:
                del rr_rem[pid]
        elif run_q == 1 and rr_rem[pid] == 0:
            queues[1].pop(0); queues[1].append(pid)
            rr_rem[pid] = quantum

    if not ticks:
        return []
    segs, cur, cs = [], ticks[0][0], ticks[0][1]
    for i in range(1, len(ticks)):
        p, tt = ticks[i]
        if p != cur or tt != ticks[i - 1][1] + 1:
            segs.append((cur, cs, ticks[i - 1][1] + 1)); cur, cs = p, tt
    segs.append((cur, cs, ticks[-1][1] + 1))
    return _merge(segs)


print("All scheduling algorithms defined.")


All scheduling algorithms defined.


In [9]:
_PAL = [
    '#4e79a7','#f28e2b','#e15759','#76b7b2',
    '#59a14f','#edc948','#b07aa1','#ff9da7',
    '#9c755f','#bab0ac','#86bcb6','#d4a5a5',
    '#2ca02c','#d62728','#9467bd','#8c564b',
]


def _timeline_ticks(timeline):
    if not timeline:
        return []
    max_t = max(e for _, _, e in timeline)
    ticks = [None] * max_t
    for pid, s, e in timeline:
        for t in range(s, e):
            ticks[t] = pid
    return ticks


def _build_state_trace(timeline, df):
    pids = df['pid'].tolist()
    arrival = {row['pid']: int(row['arrival_time']) for _, row in df.iterrows()}
    burst = {row['pid']: int(row['burst_time']) for _, row in df.iterrows()}
    ticks = _timeline_ticks(timeline)
    max_t = len(ticks)

    next_use = {pid: [float('inf')] * (max_t + 1) for pid in pids}
    next_seen = {pid: float('inf') for pid in pids}
    for t in range(max_t - 1, -1, -1):
        next_seen[ticks[t]] = t
        for pid in pids:
            next_use[pid][t] = next_seen[pid]

    executed = {pid: 0 for pid in pids}
    states = []
    for t in range(max_t + 1):
        current = ticks[t] if t < max_t else None
        arrived = [pid for pid in pids if arrival[pid] <= t]
        pending = [pid for pid in pids if arrival[pid] > t]
        completed = [pid for pid in pids if executed[pid] >= burst[pid]]
        ready = [
            pid for pid in arrived
            if executed[pid] < burst[pid] and pid != current
        ]
        ready.sort(key=lambda pid: (next_use[pid][t], arrival[pid], pid))
        remaining = {
            pid: burst[pid] - executed[pid]
            for pid in arrived
            if executed[pid] < burst[pid]
        }
        states.append({
            'time': t,
            'running': current,
            'arrived': arrived,
            'pending': pending,
            'ready': ready,
            'completed': completed,
            'remaining': remaining,
        })
        if current is not None:
            executed[current] += 1
    return states


def _gantt_style(title, df, max_t):
    pids = df['pid'].tolist()
    cmap = {p: _PAL[i % len(_PAL)] for i, p in enumerate(pids)}
    fig_w = max(22, max_t * 0.55 + 12)
    fig_h = max(5, len(pids) * 0.8 + 2)
    y_pos = {pid: i for i, pid in enumerate(reversed(pids))}
    return pids, cmap, fig_w, fig_h, y_pos


def _draw_state_panel(ax, state):
    ax.clear()
    ax.axis('off')

    def _fmt(items):
        return ', '.join(items) if items else '-'

    remaining = _fmt([f"{pid}:{state['remaining'][pid]}" for pid in state['remaining']])
    lines = [
        'Scheduler State',
        f"Time: {state['time']}",
        f"Running: {state['running'] or '-'}",
        f"Ready Structure: {_fmt(state['ready'])}",
        f"Completed: {_fmt(state['completed'])}",
        f"Pending Arrival: {_fmt(state['pending'])}",
        f"Remaining Burst: {remaining}",
    ]

    ax.text(
        0.03,
        0.98,
        '\n\n'.join(lines),
        va='top',
        ha='left',
        fontsize=17,
        color='#222',
        linespacing=1.55,
        bbox=dict(boxstyle='round,pad=0.9', facecolor='#f7f7f7', edgecolor='#cccccc'),
    )


def _draw_gantt_frame(ax, timeline, title, df, max_t, current_t=None):
    """Draw a full or partial Gantt chart up to current_t."""
    pids, cmap, _, _, y_pos = _gantt_style(title, df, max_t)
    bar_h = 0.55
    active_pid = None

    for pid, s, e in timeline:
        end = e if current_t is None else min(e, current_t)
        if end <= s:
            continue
        y = y_pos[pid]
        is_active = current_t is not None and s <= current_t < e
        face = cmap[pid]
        edge = '#111' if is_active else '#333'
        lw = 1.8 if is_active else 0.8
        alpha = 1.0 if is_active else 0.9
        ax.broken_barh(
            [(s, end - s)],
            (y - bar_h / 2, bar_h),
            facecolors=face,
            edgecolors=edge,
            linewidth=lw,
            alpha=alpha,
        )
        if end == e and e - s > 0:
            ax.text(
                (s + e) / 2,
                y,
                f'{s}-{e}',
                ha='center',
                va='center',
                fontsize=7.5,
                fontweight='bold',
                color='#111',
            )
        if is_active:
            active_pid = pid

    ax.set_xlim(-0.3, max_t + 0.8)
    ax.set_ylim(-0.8, len(pids) - 0.2)
    ax.set_yticks(list(y_pos.values()))
    ax.set_yticklabels(list(reversed(pids)), fontsize=11)
    ax.set_xticks(range(0, max_t + 2))
    ax.tick_params(axis='x', labelsize=8)
    ax.set_xlabel('Time', fontsize=11)
    ax.set_ylabel('Process', fontsize=11)
    ax.grid(axis='x', linestyle='--', alpha=0.35, color='#888')
    ax.set_axisbelow(True)
    for tick in range(0, max_t + 2):
        ax.axvline(tick, color='#ddd', linewidth=0.4, zorder=0)

    time_label = 'Complete' if current_t is None or current_t >= max_t else f'Time = {current_t}'
    active_label = f' | Running: {active_pid}' if active_pid else ''
    ax.set_title(f'{title}\n{time_label}{active_label}', fontsize=13, fontweight='bold', pad=10)

    handles = [mpatches.Patch(color=cmap[p], label=p) for p in pids]
    ax.legend(
        handles=handles,
        loc='upper right',
        fontsize=9,
        title='Process',
        framealpha=0.85,
        ncol=max(1, len(pids) // 5),
    )


def plot_gantt(timeline, title, df):
    """Static Gantt chart with a final scheduler state panel."""
    max_t = max(e for _, _, e in timeline)
    states = _build_state_trace(timeline, df)
    _, _, fig_w, fig_h, _ = _gantt_style(title, df, max_t)
    fig, (ax, ax_state) = plt.subplots(
        1,
        2,
        figsize=(fig_w, fig_h),
        gridspec_kw={'width_ratios': [3.3, 3.0]},
    )
    _draw_gantt_frame(ax, timeline, title, df, max_t)
    _draw_state_panel(ax_state, states[-1])
    plt.tight_layout()
    out = 'gantt_' + title.replace(' ', '_')[:18] + '.png'
    plt.savefig(out, dpi=140, bbox_inches='tight')
    plt.show()
    print(f'  Saved -> {out}')


def animate_gantt(timeline, title, df, interval=700, save_gif=False):
    """Animate the scheduler timeline and state trace inside the notebook."""
    if not timeline:
        print('No timeline to animate.')
        return None

    max_t = max(e for _, _, e in timeline)
    states = _build_state_trace(timeline, df)
    _, _, fig_w, fig_h, _ = _gantt_style(title, df, max_t)
    fig, (ax, ax_state) = plt.subplots(
        1,
        2,
        figsize=(fig_w, fig_h),
        gridspec_kw={'width_ratios': [3.3, 3.0]},
    )
    frames = list(range(0, max_t + 1))

    def _update(current_t):
        ax.clear()
        _draw_gantt_frame(ax, timeline, title, df, max_t, current_t=current_t)
        _draw_state_panel(ax_state, states[current_t])
        return ax.patches + ax.texts + ax_state.texts

    anim = FuncAnimation(fig, _update, frames=frames, interval=interval, repeat=False, blit=False)
    display(HTML(anim.to_jshtml()))

    if save_gif:
        out = 'gantt_' + title.replace(' ', '_')[:18] + '.gif'
        anim.save(out, writer=PillowWriter(fps=max(1, int(1000 / interval))))
        print(f'  Saved animation -> {out}')

    plt.close(fig)
    return anim


def compute_stats(timeline, df):
    """Compute and print per-process and average scheduling metrics."""
    rows = []
    for _, row in df.iterrows():
        pid = row['pid']
        segs = [(s, e) for p, s, e in timeline if p == pid]
        if not segs:
            continue
        fs = min(s for s, e in segs)
        fe = max(e for s, e in segs)
        rt = fs - row['arrival_time']
        tat = fe - row['arrival_time']
        wt = tat - row['burst_time']
        rows.append({
            'PID': pid,
            'Arrival': int(row['arrival_time']),
            'Burst': int(row['burst_time']),
            'Start': int(fs),
            'Finish': int(fe),
            'Response Time': int(rt),
            'Waiting Time': int(wt),
            'Turnaround Time': int(tat),
        })
    stats = pd.DataFrame(rows)
    avg = stats[['Response Time', 'Waiting Time', 'Turnaround Time']].mean()
    sep = '-' * 48
    print(f'\n{sep}')
    print(f'  Avg Response Time   : {avg["Response Time"]:.2f}')
    print(f'  Avg Waiting Time    : {avg["Waiting Time"]:.2f}')
    print(f'  Avg Turnaround Time : {avg["Turnaround Time"]:.2f}')
    print(sep)
    return stats


print('Gantt, animation, and statistics functions defined.')


Gantt, animation, and statistics functions defined.


In [10]:
# =============================================================
# configuration
# =============================================================
ALGORITHM      = 'RR'   # FCFS | SJF | SRTF | RR | PRIORITY | PRIORITY_P | MLQ
QUANTUM        = 3            # time quantum used by RR and MLQ (queue 1)
ANIMATE        = True         # show a step-by-step animation with a live state panel
FRAME_INTERVAL = 300          # milliseconds per animation frame
SAVE_GIF       = True        # optionally export the animation as a GIF

LABELS = {
    'FCFS':       'First Come First Served (Non-Preemptive)',
    'SJF':        'Shortest Job First (Non-Preemptive)',
    'SRTF':       'Shortest Remaining Time First (Preemptive)',
    'RR':         f'Round Robin  [q={QUANTUM}]',
    'PRIORITY':   'Priority Scheduling (Non-Preemptive)',
    'PRIORITY_P': 'Priority Scheduling (Preemptive)',
    'MLQ':        f'Multilevel Queue  [Q0=FCFS | Q1=RR q={QUANTUM} | Q2=FCFS]',
}
ALGOS = {
    'FCFS':       lambda p: fcfs(p),
    'SJF':        lambda p: sjf(p),
    'SRTF':       lambda p: srtf(p),
    'RR':         lambda p: round_robin(p, QUANTUM),
    'PRIORITY':   lambda p: priority_np(p),
    'PRIORITY_P': lambda p: priority_p(p),
    'MLQ':        lambda p: multilevel_queue(p, QUANTUM),
}

if ALGORITHM not in ALGOS:
    print(f"Unknown algorithm '{ALGORITHM}'. Valid: {', '.join(ALGOS)}")
else:
    procs    = df.to_dict('records')
    label    = LABELS[ALGORITHM]
    print(f'Running: {label}\n')
    timeline = ALGOS[ALGORITHM](deepcopy(procs))
    if ANIMATE:
        animate_gantt(timeline, label, df, interval=FRAME_INTERVAL, save_gif=SAVE_GIF)
    else:
        plot_gantt(timeline, label, df)
    stats = compute_stats(timeline, df)
    display(stats)


Running: Round Robin  [q=3]



  Saved animation -> gantt_Round_Robin__[q=3].gif

------------------------------------------------
  Avg Response Time   : 8.25
  Avg Waiting Time    : 19.75
  Avg Turnaround Time : 24.50
------------------------------------------------


,PID,Arrival,Burst,Start,Finish,Response Time,Waiting Time,Turnaround Time
0,P1,0,8,0,32,0,24,32
1,P2,1,4,3,24,2,19,23
2,P3,2,9,6,38,4,27,36
3,P4,3,5,9,30,6,22,27
4,P5,4,2,15,17,11,11,13
5,P6,5,6,17,35,12,24,30
6,P7,6,3,20,23,14,14,17
7,P8,7,1,24,25,17,17,18
